# Schedule Adherence at Timepoint

### Definition
At each transit stop, the proportion of trips that are less than five minutes late and less than one minute early.

### Reporting Grain
Reporting Period and Day Part. (also mapped at timepoints here)

# Average Deviation from Schedule

### Definition

The difference between when the bus is scheduled to arrive at each stop and when the bus arrives based on real time data. Reported as 50th and 80th percentile.



In [1]:
import os
from sqlalchemy import create_engine
from jinja2 import Environment, FileSystemLoader
import pandas as pd
import geopandas as gpd
import shapely.wkt

from src.data_loading import (
    get_matching_shapes_feeds,
    get_schedule_feed_entries,
    get_schedule_rt_stop_times,
)
from src.metric_calculation import calculate_on_time_performance

### Parameters

In [2]:
TARGET_DATES = [
    "2026-06-15","2026-06-16","2026-06-17","2026-06-18","2026-06-19"
]
TARGET_ROUTE_SHORT_NAME = os.environ.get("TARGET_ROUTE_SHORT_NAME", "ECR")
TARGET_SCHEDULE_FEED = os.environ.get("TARGET_SCHEDULE_FEED", "Bay Area 511 SamTrans Schedule")
CONSTANT_MACROS = {
    "TARGET_DATES": TARGET_DATES,
    "TARGET_ROUTE_SHORT_NAME": TARGET_ROUTE_SHORT_NAME,
    "TARGET_SCHEDULE_FEED": TARGET_SCHEDULE_FEED
}
CONSTANT_MACROS

{'TARGET_DATES': ['2026-06-15',
  '2026-06-16',
  '2026-06-17',
  '2026-06-18',
  '2026-06-19'],
 'TARGET_ROUTE_SHORT_NAME': 'ECR',
 'TARGET_SCHEDULE_FEED': 'Bay Area 511 SamTrans Schedule'}

In [3]:
# min count of trips per service period to be considered reliable
MIN_TRIPS = 10

# beat the published arrival by more than this and the vehicle is early; miss it
# by more than this and it is late. Both strict
EARLY_SECONDS = 60
LATE_SECONDS = 5 * 60

### Implementation

In [4]:
# gcp boilerplate
CALITP_BQ_MAX_BYTES = os.environ.get("CALITP_BQ_MAX_BYTES", 5_000_000_000)
CALITP_BQ_LOCATION = os.environ.get("CALITP_BQ_LOCATION", "us-west2")
engine = create_engine(
    f"bigquery://cal-itp-data-infra/?maximum_bytes_billed={CALITP_BQ_MAX_BYTES}",  # noqa: E231
    location=CALITP_BQ_LOCATION
)
env = Environment(loader=FileSystemLoader("templates"))

In [5]:
# Get the target route's service patterns, their geometry, and the feeds behind them
matching_shapes, feed_dates, feed_keys = get_matching_shapes_feeds(
    engine, env, CONSTANT_MACROS, min_trips=MIN_TRIPS
)
display(feed_dates)
matching_shapes

,service_date,feed_key
0,2026-06-19,d71492afe126cd61f629b17c3c056fcd
1,2026-06-16,d71492afe126cd61f629b17c3c056fcd
2,2026-06-17,d71492afe126cd61f629b17c3c056fcd
3,2026-06-18,d71492afe126cd61f629b17c3c056fcd
5,2026-06-15,4dfefaf518766d1cfc3d254a65e4295e


,feed_key,last_seen_date,gtfs_dataset_name,shape_id,route_id,route_short_name,ct,geometry
0,4dfefaf518766d1cfc3d254a65e4295e,2026-06-15,Bay Area 511 SamTrans Schedule,ECR2068,ECR,ECR,350,"LINESTRING (-122.46879 37.70620, -122.46897 37..."
1,4dfefaf518766d1cfc3d254a65e4295e,2026-06-15,Bay Area 511 SamTrans Schedule,ECR2069,ECR,ECR,350,"LINESTRING (-122.16599 37.44395, -122.16578 37..."
2,d71492afe126cd61f629b17c3c056fcd,2026-06-19,Bay Area 511 SamTrans Schedule,ECR2069,ECR,ECR,350,"LINESTRING (-122.16599 37.44395, -122.16578 37..."
3,d71492afe126cd61f629b17c3c056fcd,2026-06-19,Bay Area 511 SamTrans Schedule,ECR2068,ECR,ECR,350,"LINESTRING (-122.46879 37.70620, -122.46897 37..."


In [6]:
# Get the schedule feed entry for every feed that was active during the window
feed_dates = get_schedule_feed_entries(engine, env, feed_dates)
feed_dates

,service_date,feed_key,base64_url,_valid_from
0,2026-06-19,d71492afe126cd61f629b17c3c056fcd,aHR0cHM6Ly9hcGkuNTExLm9yZy90cmFuc2l0L2RhdGFmZW...,2026-06-16 03:00:00.395507+00:00
1,2026-06-16,d71492afe126cd61f629b17c3c056fcd,aHR0cHM6Ly9hcGkuNTExLm9yZy90cmFuc2l0L2RhdGFmZW...,2026-06-16 03:00:00.395507+00:00
2,2026-06-17,d71492afe126cd61f629b17c3c056fcd,aHR0cHM6Ly9hcGkuNTExLm9yZy90cmFuc2l0L2RhdGFmZW...,2026-06-16 03:00:00.395507+00:00
3,2026-06-18,d71492afe126cd61f629b17c3c056fcd,aHR0cHM6Ly9hcGkuNTExLm9yZy90cmFuc2l0L2RhdGFmZW...,2026-06-16 03:00:00.395507+00:00
4,2026-06-15,4dfefaf518766d1cfc3d254a65e4295e,aHR0cHM6Ly9hcGkuNTExLm9yZy90cmFuc2l0L2RhdGFmZW...,2026-05-16 03:00:00.573378+00:00


In [7]:
# One row per trip and timepoint: when the vehicle actually arrived, and the
# arrival dim_stop_arrivals published for that trip at that stop. This is the
# same frame trip_duration_variability.ipynb reads - that metric uses only the
# realtime columns, this one compares the two arrivals
schedule_rt_stop_times = get_schedule_rt_stop_times(
    engine,
    env,
    CONSTANT_MACROS,
    feed_dates=feed_dates,
    shape_ids=matching_shapes.shape_id.to_list(),
)
schedule_rt_stop_times

,service_date,trip_id,trip_key,stop_id,feed_key,shape_id,stop_sequence,actual_arrival_pacific,actual_departure_pacific,n_predictions,scheduled_arrival_sec,scheduled_arrival_pacific,pt_geom,arrival_deviation_seconds
0,2026-06-15,15273728-154-Blocks-Weekday-60,065c8c87958002e551da443d52c40213,333505,4dfefaf518766d1cfc3d254a65e4295e,ECR2068,8,2026-06-15 17:08:00,2026-06-15 17:08:00,16,61680,2026-06-15 17:08:00,POINT(-122.465753 37.684618),0
1,2026-06-15,15273728-154-Blocks-Weekday-60,065c8c87958002e551da443d52c40213,334087,4dfefaf518766d1cfc3d254a65e4295e,ECR2068,13,2026-06-15 17:15:46,2026-06-15 17:15:46,37,62206,2026-06-15 17:16:46,POINT(-122.446544 37.66342),-60
2,2026-06-15,15273728-154-Blocks-Weekday-60,065c8c87958002e551da443d52c40213,334410,4dfefaf518766d1cfc3d254a65e4295e,ECR2068,14,2026-06-15 17:17:46,2026-06-15 17:17:46,43,62266,2026-06-15 17:17:46,POINT(-122.443469 37.661748),0
3,2026-06-15,15273728-154-Blocks-Weekday-60,065c8c87958002e551da443d52c40213,334077,4dfefaf518766d1cfc3d254a65e4295e,ECR2068,16,2026-06-15 17:21:20,2026-06-15 17:21:20,55,62420,2026-06-15 17:20:20,POINT(-122.436081 37.656146),60
4,2026-06-15,15273728-154-Blocks-Weekday-60,065c8c87958002e551da443d52c40213,334090,4dfefaf518766d1cfc3d254a65e4295e,ECR2068,18,2026-06-15 17:22:48,2026-06-15 17:22:48,58,62568,2026-06-15 17:22:48,POINT(-122.43055 37.649842),0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14928,2026-06-18,15273781-154-Blocks-Weekday-60,fed1c437218f5f86dedc7795c0568606,345400,d71492afe126cd61f629b17c3c056fcd,ECR2068,76,2026-06-18 17:50:04,2026-06-18 17:50:04,90,64024,2026-06-18 17:47:04,POINT(-122.208728 37.466759),180
14929,2026-06-18,15273781-154-Blocks-Weekday-60,fed1c437218f5f86dedc7795c0568606,345400,d71492afe126cd61f629b17c3c056fcd,ECR2068,76,2026-06-18 17:43:04,2026-06-18 17:43:04,90,64024,2026-06-18 17:47:04,POINT(-122.208728 37.466759),-240
14930,2026-06-18,15273781-154-Blocks-Weekday-60,fed1c437218f5f86dedc7795c0568606,346045,d71492afe126cd61f629b17c3c056fcd,ECR2068,80,2026-06-18 17:50:55,2026-06-18 17:50:55,90,64975,2026-06-18 18:02:55,POINT(-122.177694 37.450628),-720
14931,2026-06-18,15273781-154-Blocks-Weekday-60,fed1c437218f5f86dedc7795c0568606,347017,d71492afe126cd61f629b17c3c056fcd,ECR2068,81,2026-06-18 17:56:13,2026-06-18 17:56:13,90,65293,2026-06-18 18:08:13,POINT(-122.168385 37.444092),-720


In [8]:
# How much of the realtime data found a published arrival to be measured
# against. The join is a left one, so an unmatched trip arrives here as a null
# deviation rather than dropping out - the on time metric drops those, the
# travel time metric keeps them
n_arrivals = len(schedule_rt_stop_times)
n_matched = schedule_rt_stop_times["arrival_deviation_seconds"].notna().sum()
print(f"{n_matched:,} of {n_arrivals:,} timepoint arrivals matched a published arrival "
      f"({n_matched / n_arrivals:.1%})")

# the deviation before any bucketing, in minutes. A long tail either side is
# worth a look before trusting the proportions: a cluster near a whole number of
# hours usually means a timezone or a GTFS-hours-past-24 problem rather than
# service running that badly
(schedule_rt_stop_times["arrival_deviation_seconds"]).describe()

14,933 of 14,933 timepoint arrivals matched a published arrival (100.0%)


count    14933.000000
mean       157.837005
std        409.568463
min      -1440.000000
25%          0.000000
50%        120.000000
75%        300.000000
max       3720.000000
Name: arrival_deviation_seconds, dtype: float64

### Results

In [9]:
# Both tables at once, pooled over every selected date.
#
# by_stop says where along the line punctuality goes: a trip that leaves on time
# and loses six minutes by the end of the run shows up at the last timepoints
# only. by_period says how the whole pattern did in a day part - its percentiles
# are taken over the arrivals rather than over the per-stop numbers, so they are
# not the average of the table above.
on_time = calculate_on_time_performance(
    schedule_rt_stop_times,
    early_seconds=EARLY_SECONDS,
    late_seconds=LATE_SECONDS,
)
on_time.by_period

,shape_id,time_period,n_timepoint_arrivals,p50_absolute_deviation_seconds,p80_absolute_deviation_seconds,proportion_early,proportion_late,proportion_not_on_time
0,ECR2068,Early AM,628,120.0,360.0,0.074841,0.200637,0.275478
1,ECR2068,AM Peak,1188,180.0,420.0,0.114478,0.257576,0.372054
2,ECR2068,Midday,1960,180.0,480.0,0.192347,0.205612,0.397959
3,ECR2068,PM Peak,2520,180.0,492.0,0.245635,0.232143,0.477778
4,ECR2068,Evening,1292,180.0,540.0,0.267802,0.210526,0.478328
5,ECR2069,Early AM,662,120.0,360.0,0.061934,0.203927,0.265861
6,ECR2069,AM Peak,1080,180.0,360.0,0.100926,0.225000,0.325926
7,ECR2069,Midday,1859,240.0,480.0,0.098440,0.293706,0.392146
8,ECR2069,PM Peak,2517,180.0,360.0,0.139849,0.181565,0.321414
9,ECR2069,Evening,1227,180.0,360.0,0.146699,0.194784,0.341483


In [10]:
on_time.by_stop

,shape_id,stop_sequence,stop_id,time_period,n_timepoint_arrivals,p50_absolute_deviation_seconds,p80_absolute_deviation_seconds,proportion_early,proportion_late,proportion_not_on_time
0,ECR2068,1,332014,Early AM,16,60.0,60.0,0.000000,0.000000,0.000000
1,ECR2068,1,332014,AM Peak,19,120.0,180.0,0.000000,0.052632,0.052632
2,ECR2068,1,332014,Midday,10,90.0,180.0,0.000000,0.100000,0.100000
3,ECR2068,1,332014,PM Peak,16,120.0,300.0,0.000000,0.187500,0.187500
4,ECR2068,1,332014,Evening,10,180.0,432.0,0.000000,0.300000,0.300000
...,...,...,...,...,...,...,...,...,...,...
330,ECR2069,75,332019,Early AM,12,240.0,456.0,0.166667,0.416667,0.583333
331,ECR2069,75,332019,AM Peak,30,240.0,372.0,0.533333,0.100000,0.633333
332,ECR2069,75,332019,Midday,57,240.0,420.0,0.631579,0.122807,0.754386
333,ECR2069,75,332019,PM Peak,70,330.0,540.0,0.600000,0.128571,0.728571


In [13]:
# Map of one day part
#
# Both service patterns are drawn as plain lines for context. The metric lives on
# the stops, coloured by the share of arrivals that were not on time.
MAP_TIME_PERIOD = "Midday"

# The stop times query carries each stop's point geometry, so the metric maps
# straight onto it - no projection onto the shape is needed the way it is for
# the segment metric in trip_duration_variability
stops = (
    schedule_rt_stop_times[["shape_id", "stop_sequence", "stop_id", "pt_geom"]]
    .drop_duplicates()
    .dropna(subset=["pt_geom"])
)
stops_gdf = gpd.GeoDataFrame(
    stops.drop(["pt_geom"], axis=1),
    geometry=gpd.GeoSeries(stops["pt_geom"].map(shapely.wkt.loads), crs=4326),
)

stops_for_period = stops_gdf.merge(
    on_time.by_stop.loc[on_time.by_stop["time_period"] == MAP_TIME_PERIOD],
    on=["shape_id", "stop_sequence", "stop_id"],
    how="inner",
)

# the proportions are 0 to 1; the map reads better in percent. The colour and the
# tooltip use the one column, so the legend matches what is hovered
for column in ["proportion_early", "proportion_late", "proportion_not_on_time"]:
    stops_for_period[column] = (stops_for_period[column] * 100).round(1)
for column in [
    "p50_absolute_deviation_seconds",
    "p80_absolute_deviation_seconds",
]:
    stops_for_period[column] = stops_for_period[column].round(1)

# the shapes underneath, uncoloured. a shape carries the same geometry in every
# feed version it appears in, so draw each one once
shape_lines = matching_shapes.drop_duplicates(subset=["shape_id"])[
    ["shape_id", "geometry"]
]
period_map = shape_lines.explore(
    color="#888888",
    tooltip=["shape_id"],
    tiles="CartoDB positron",
    name="service patterns",
)
stops_for_period.explore(
    m=period_map,
    column="proportion_not_on_time",
    cmap="plasma",
    marker_kwds={"radius": 6},
    style_kwds={"stroke": False, "fillOpacity": 0.9},
    tooltip=[
        "shape_id",
        "stop_sequence",
        "stop_id",
        "n_timepoint_arrivals",
        "p50_absolute_deviation_seconds",
        "p80_absolute_deviation_seconds",
        "proportion_early",
        "proportion_late",
        "proportion_not_on_time",
    ],
    tooltip_kwds={
        "aliases": [
            "Shape",
            "Stop sequence",
            "Stop",
            "Arrivals",
            "Median deviation (s)",
            "80th pct deviation (s)",
            "% early",
            "% late",
            "% not on time",
        ]
    },
    legend_kwds={
        "caption": f"% of {MAP_TIME_PERIOD.lower()} arrivals not on time"
    },
    name=f"{MAP_TIME_PERIOD} on time performance",
)
period_map.save(f"on_time_performance_{MAP_TIME_PERIOD.lower()}.html")
period_map